In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [28]:
%pip install -U dagshub mlflow skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [29]:
import mlflow
import os

os.environ['MLFLOW_TRACKING_USERNAME'] = 'izere23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '90676af55d048ee40dd81f3034e9b5d260734931'

mlflow.set_tracking_uri("https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow")

In [43]:
# Cell 5: Imputation function
def impute(test_refined):
    num_medians = {
        'MSSubClass': 50.0,
        'LotFrontage': 70.0,
        'LotArea': 9600.0,
        'OverallQual': 6.0,
        'OverallCond': 5.0,
        'YearBuilt': 1972.0,
        'YearRemodAdd': 1994.0,
        'MasVnrArea': 0.0,
        'BsmtFinSF1': 384.5,
        'BsmtFinSF2': 0.0,
        'BsmtUnfSF': 480.0,
        'TotalBsmtSF': 997.5,
        '1stFlrSF': 1095.0,
        '2ndFlrSF': 0.0,
        'GrLivArea': 1473.0,
        'BsmtFullBath': 0.0,
        'BsmtHalfBath': 0.0,
        'FullBath': 2.0,
        'HalfBath': 0.0,
        'BedroomAbvGr': 3.0,
        'TotRmsAbvGrd': 6.0,
        'Fireplaces': 1.0,
        'GarageYrBlt': 1980.0,
        'GarageCars': 2.0,
        'GarageArea': 482.0,
        'WoodDeckSF': 0.0,
        'OpenPorchSF': 27.0,
        'EnclosedPorch': 0.0,
        'ScreenPorch': 0.0,
        'MoSold': 6.0,
        'YrSold': 2008.0
    }

    cat_modes = {
        'MSZoning': 'RL',
        'LotShape': 'Reg',
        'LandContour': 'Lvl',
        'LotConfig': 'Inside',
        'LandSlope': 'Gtl',
        'Neighborhood': 'NAmes',
        'Condition1': 'Norm',
        'BldgType': '1Fam',
        'HouseStyle': '1Story',
        'RoofStyle': 'Gable',
        'Exterior1st': 'VinylSd',
        'Exterior2nd': 'VinylSd',
        'MasVnrType': 'BrkFace',
        'ExterQual': 'TA',
        'ExterCond': 'TA',
        'Foundation': 'PConc',
        'BsmtQual': 'TA',
        'BsmtCond': 'TA',
        'BsmtExposure': 'No',
        'BsmtFinType1': 'Unf',
        'BsmtFinType2': 'Unf',
        'HeatingQC': 'Ex',
        'CentralAir': 'Y',
        'Electrical': 'SBrkr',
        'KitchenQual': 'TA',
        'Functional': 'Typ',
        'FireplaceQu': 'Gd',
        'GarageType': 'Attchd',
        'GarageFinish': 'Unf',
        'GarageQual': 'TA',
        'GarageCond': 'TA',
        'PavedDrive': 'Y',
        'SaleType': 'WD',
        'SaleCondition': 'Normal'
    }

    for col, val in num_medians.items():
        if col in test_refined.columns:
            test_refined[col] = test_refined[col].fillna(val)

    for col, val in cat_modes.items():
        if col in test_refined.columns:
            test_refined[col] = test_refined[col].fillna(val)

    return test_refined

In [44]:
test_raw = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
submission_ids = test_raw['Id']

cols_to_drop = [
    'Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence',
    'Utilities', 'Street', 'PoolArea', 'Condition2',
    'RoofMatl', '3SsnPorch', 'LowQualFinSF',
    'Heating', 'MiscVal', 'KitchenAbvGr'
]
print(test_raw.shape)
test_refined = test_raw.drop(columns=cols_to_drop)
print(test_raw.shape)
test_refined = impute(test_refined)

num_cols = [col for col in test_refined.columns if test_refined[col].dtype != 'object']
cat_cols = [col for col in test_refined.columns if test_refined[col].dtype == 'object']

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = encoder.fit_transform(test_refined[cat_cols])

encoded_cols = encoder.get_feature_names_out(cat_cols)
encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=test_refined.index)

X_test = pd.concat([test_refined[num_cols], encoded_df], axis=1)

print(X_test.shape)

(1459, 80)
(1459, 80)
(1459, 237)


In [ ]:
import mlflow.xgboost

model = mlflow.xgboost.load_model("models:/xgboost_model/1")
print("Model loaded!")

train_cols = model.get_booster().feature_names
X_test = X_test.reindex(columns=train_cols, fill_value=0)

final_preds = model.predict(X_test)
print(final_preds[:10])

submission = pd.DataFrame({
    'Id': submission_ids,
    'SalePrice': final_preds
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("Done!")
print(submission.head())